# Lab 11: Government Data APIs

*INFO 4617/5617 · Week 11 · Chapter 11, [Government Data APIs](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html)*

Today we code together. Each step below says what we're building and why, and its code cell holds only the comments that frame it: we write the code live, in class. Nothing here is turned in.

We'll answer one question with two public APIs: **how did Boulder's housing come to look the way it does?** The Census Bureau tells us what the housing stock is today; FRED, the Federal Reserve Bank of St. Louis's data service, tells us about the economy it was built in.

| Step | What we build | Minutes |
|---|---|---|
| 0 | Setup: keys from the environment, never in the notebook | 5 |
| 1 | Find a variable in the Census catalog | 8 |
| 2–4 | Query Boulder, compare it with Longmont, and sanity-check a number | 22 |
| 5–7 | FRED series, a transformation the server does for us, and Denver against the nation | 25 |
| 8 | Put the two together | 10 |
| 9 | Optional: campaign finance from the FEC | — |
| 10 | What did we learn? | 5 |

**Before class:** request free keys from the [Census Bureau](https://api.census.gov/data/key_signup.html) and [FRED](https://fred.stlouisfed.org/docs/api/api_key.html) (and, for Step 9, the [FEC](https://api.open.fec.gov/developers/)). Set them as the environment variables `CENSUS_API_KEY` and `FRED_API_KEY` before you start Jupyter; *Missing Manual* Chapter 34 shows how. Approval can take a few minutes, so don't wait until class.

## Step 0 — Set up, and keep the keys out of the notebook (5 minutes)

An API key identifies you, counts your requests, and is yours alone. A key typed into a notebook travels with the notebook: into a GitHub repository, a class submission, a classmate's laptop. Read the keys from environment variables instead, and stop with a clear message if one is missing.

In [ ]:
# Import os, requests, pandas (as pd), and matplotlib.pyplot (as plt)
# Read CENSUS_API_KEY and FRED_API_KEY from the environment (os.environ.get)
# If either is missing, raise an error that says which one, and how to set it
# Your code here

## Step 1 — Find a variable in the Census catalog (8 minutes)

The American Community Survey has thousands of variables, organized in groups. Group `B25034` is "Year Structure Built". Its catalog is itself an API: ask for the group's JSON and print each variable's code and label. Codes ending in `E` are estimates; those ending in `M` are margins of error, so skip them. Browse the other groups at [api.census.gov/data/2022/acs/acs5/groups.html](https://api.census.gov/data/2022/acs/acs5/groups.html).

In [ ]:
# Request https://api.census.gov/data/2022/acs/acs5/groups/B25034.json
# For each variable, in order, print its code and label, skipping the margins of error
# Your code here

## Step 2 — Ask about one place (8 minutes)

Places are named by FIPS codes: Colorado is state `08`, and Boulder is place `07850`. Ask for the place's name and the first few `B25034` estimates. The Census answers with a list of lists whose first row is the header, and every value is a string: make a DataFrame, and turn the numbers into numbers.

In [ ]:
# census_url = "https://api.census.gov/data/2022/acs/acs5"
# params: get="NAME,B25034_001E,...,B25034_005E", for="place:07850", in="state:08", key=census_key
# Make a DataFrame from rows 1 onward, with row 0 as the columns, and convert the B25034 columns to numbers
# Your code here

## Step 3 — Compare two places (7 minutes)

Several places can go in one request: `place:07850,45970` is Boulder and Longmont. Plot the two cities' housing by decade built, side by side.

In [ ]:
# Ask again with for="place:07850,45970"
# Plot the decades (002E through 005E) as grouped bars, one color per city
# Your code here

## Step 4 — Sanity-check a number (7 minutes)

A variable with the right-sounding label can still be the wrong one. Before trusting a pipeline, check one value you can judge. Ask for Boulder's median household income (`B19013_001E`). Does it look like a city's median income: tens of thousands of dollars, not 7 or 70 million?

In [ ]:
# Ask for NAME and B19013_001E for Boulder (place:07850 in state:08)
# Print the city and its median household income, formatted with a dollar sign and commas
# Your code here

## Step 5 — A FRED series (8 minutes)

FRED holds hundreds of thousands of time series, each with an ID you find by searching [fred.stlouisfed.org](https://fred.stlouisfed.org/). `MHICO08013A052NCEN` is median household income for Boulder County, from the Census Bureau's SAIPE program; the ID embeds the state (`CO`) and the county's FIPS code (`08013`). Ask for its observations. FRED writes a missing value as `"."`, so convert with `errors="coerce"`.

In [ ]:
# fred_url = "https://api.stlouisfed.org/fred/series/observations"
# params: series_id, api_key=fred_key, file_type=json, observation_end="2026-09-30"
# Make a DataFrame from ["observations"]; convert date to dates and value to numbers; show the last rows
# Your code here

## Step 6 — Let the server do the math (7 minutes)

FRED can transform a series before it sends it. `units="pc1"` returns the percent change from a year before, so the Consumer Price Index (`CPIAUCSL`) comes back as the inflation rate. Plot it since 2020, with a line at the Federal Reserve's 2% target.

In [ ]:
# Ask for CPIAUCSL with units="pc1" and observation_start="2020-01-01"
# Plot it, with a dashed horizontal line at 2
# Your code here

## Step 7 — Denver against the nation (10 minutes)

National numbers hide local ones. `CUUSA433SA0` is the CPI for the Denver area. The Bureau of Labor Statistics publishes it once a year, and it isn't seasonally adjusted, so compare it with the national series that isn't either, `CPIAUCNS`, which FRED can average into years for you (`frequency="a"`). Ask for both as year-over-year changes since 2000, merge them on date, and plot them together. The merge keeps only the years both series have.

In [ ]:
# Ask for CUUSA433SA0, and CPIAUCNS with frequency="a"; both with units="pc1" and observation_start="2000-01-01"
# Merge them on date (inner), and plot both lines with a dashed line at 2
# Your code here

## Step 8 — Put the two together (10 minutes)

The Census says what Boulder's housing is; FRED says what borrowing cost when it was built. Ask the Census for Boulder's housing by decade back to the 1970s (`B25034_002E` to `B25034_007E`), and FRED for the 30-year mortgage rate as yearly averages (`MORTGAGE30US`, `frequency="a"`) since 1970. Plot them side by side.

One trap: the Census marks a value it can't publish with a code such as `-666666666`, not a blank. Check for it before you convert.

In [ ]:
# Census: NAME and B25034_002E through B25034_007E for Boulder; turn codes starting with -6666 into missing values
# FRED: MORTGAGE30US with frequency="a" and observation_start="1970-01-01"
# Two panels: bars of housing by decade, and the mortgage rate over time
# Your code here

## Step 9 — Optional: campaign finance from the FEC (if there's time)

The Federal Election Commission's API needs a third key, `FEC_API_KEY`. Search for candidates by name, then ask for one candidate's totals: IDs start with the office sought (`P` for president, `S` for Senate, `H` for House).

In [ ]:
# Read FEC_API_KEY from the environment
# Search https://api.open.fec.gov/v1/candidates/search/ for q="Biden", office="P", election_year=2024
# Then get totals from https://api.open.fec.gov/v1/candidate/P80000722/totals/ for 2024
# Your code here

## Step 10 — What did we learn? (5 minutes)

In a few sentences, as comments or in a Markdown cell:

- Which decade built the most of Boulder's housing, and which the least? Does the mortgage rate explain it, or does something else, such as the city's limits on growth?
- When did Denver's inflation run above the nation's, and by how much at most?
- Where did each API make you look something up: a variable code, a series ID, a FIPS code? Where would you look next time?

In [ ]:
# Your answers here, as comments
